# Tabela de Lift (S3 → S4), Interpretação e Sprint Review

**Objetivo:** construir a tabela de lift comparando os baselines da Sprint 3 (features antigas) com os da Sprint 4 (features congeladas), discutir a evolução agronômica das métricas e registrar o incremento da Sprint Review com o PO.

**Entrada:** métricas do Tópico 5 (`../models/s4_metricas_baselines_teste.csv`, gerado pelo notebook 09) e baselines da Sprint 3 reproduzidos aqui, no mesmo split e com o mesmo código.  
**Saída:** tabelas de lift (teste oficial e backtest), análise de ganho ou descarte, e seção da Sprint Review, prontas para `docs/sprints/Sprint4_TrilhaC.md`.

**Regras seguidas**
- Mesmo split da Sprint 2/3 (treino 2015–2022, teste 2023–2024); o teste só é usado para `predict` e métricas, nunca para escolher nada.
- Nenhum número da Sprint 3 é colado de notebooks antigos: S3 é retreinado aqui.
- Métrica indefinida aparece como **`n/d`**, e não como 0.
- O backtest dentro do treino é **diagnóstico complementar**, não substitui o teste oficial.

In [1]:
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\josea\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Dados e split temporal (mesmo corte da Sprint 2/3)

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

sns.set_theme(style='whitegrid')

# ---------------- Configuração (igual ao notebook 09) ----------------
RANDOM_STATE = 42
ANO_CORTE_TREINO = 2022          # treino: ano <= 2022 | teste: ano >= 2023
ANO_INICIO_BACKTEST = 2019       # primeira dobra do backtest treina com 2015–2018
TARGET = 'quebra_safra'          # 1 = Quebra (< 3.400 kg/ha), 0 = Normal
CLASSES = [0, 1]
ORDEM_MODELOS = ['DummyClassifier', 'Persistência', 'GaussianNB']

FEATURES_S4 = ['deficit_hidrico_solo', 'dias_pico_calor', 'amplitude_termica', 'precipitacao_enchimento']
FEATURES_S3 = ['PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT',
               'amplitude_termica', 'precipitacao_por_temperatura', 'GWET_x_PREC']
CANDIDATAS = FEATURES_S3 + ['dias_pico_calor', 'deficit_hidrico_solo', 'veranico_max_dias', 'precipitacao_enchimento']
CANDIDATAS = list(dict.fromkeys(CANDIDATAS))   # sem duplicata (amplitude_termica aparece nos dois)

COLUNAS_PROIBIDAS = {'ano', TARGET, 'rendimento_kg_ha', 'quantidade_produzida_t',
                     'area_colhida_ha', 'area_plantada_ha'}
assert not COLUNAS_PROIBIDAS & set(FEATURES_S4), 'Vazamento: coluna proibida em FEATURES_S4'
assert not COLUNAS_PROIBIDAS & set(FEATURES_S3), 'Vazamento: coluna proibida em FEATURES_S3'

# ---------------- Dados ----------------
df = pd.read_csv('../data/processed/dados_modelo.csv').sort_values('ano').reset_index(drop=True)

necessarias = set(CANDIDATAS) | {TARGET, 'ano'}
faltando = sorted(necessarias - set(df.columns))
assert not faltando, f'Colunas ausentes em dados_modelo.csv (rode o notebook 08): {faltando}'
assert df['ano'].is_unique and (df['ano'].diff().dropna() == 1).all(), 'Série anual inconsistente'
assert df[sorted(necessarias)].isna().sum().sum() == 0, 'Há NaN nas colunas usadas'

train_mask = df['ano'] <= ANO_CORTE_TREINO
df_train, df_test = df[train_mask].copy(), df[~train_mask].copy()
assert df_train['ano'].max() < df_test['ano'].min()
y_train, y_test = df_train[TARGET], df_test[TARGET]
ROTULO_POR_ANO = df.set_index('ano')[TARGET]

print(f"Treino: {df_train['ano'].min()}–{df_train['ano'].max()} (N = {len(df_train)}) | quebras: {int(y_train.sum())}")
print(f"Teste : {df_test['ano'].min()}–{df_test['ano'].max()} (N = {len(df_test)}) | quebras: {int(y_test.sum())}")

Treino: 2015–2022 (N = 8) | quebras: 3
Teste : 2023–2024 (N = 2) | quebras: 0


## 2. Funções (mesmo pipeline congelado do notebook 09)

In [3]:
def construir_preprocessor(features):
    return ColumnTransformer(
        transformers=[('escala_climatica', StandardScaler(), list(features))],
        remainder='drop',
    )


def criar_pipeline(modelo, features):
    return Pipeline([('preprocessor', construir_preprocessor(features)), ('model', modelo)])


def prever_baselines(features, df_tr, df_te):
    # Dummy e GaussianNB treinados SÓ em df_tr; persistência = rótulo observado em t-1.
    preds, pipes = {}, {}
    modelos = {
        'DummyClassifier': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'GaussianNB': GaussianNB(),
    }
    for nome, modelo in modelos.items():
        pipe = criar_pipeline(modelo, features)
        pipe.fit(df_tr[features], df_tr[TARGET])
        preds[nome] = pipe.predict(df_te[features])
        pipes[nome] = pipe
    persistencia = df_te['ano'].map(lambda a: ROTULO_POR_ANO.get(a - 1, np.nan))
    assert persistencia.notna().all(), 'Persistência sem rótulo da safra anterior'
    preds['Persistência'] = persistencia.astype(int).to_numpy()
    return {k: preds[k] for k in ORDEM_MODELOS}, pipes


def _prf(tp, fp, fn):
    precisao = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else np.nan
    return precisao, recall, f1


def metricas_por_classe(nome, y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=CLASSES).ravel()
    p1, r1, f1_1 = _prf(tp, fp, fn)   # classe 1 = Quebra (positiva)
    p0, r0, f1_0 = _prf(tn, fn, fp)   # classe 0 = Normal
    return {
        'modelo': nome, 'N': len(y_true),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'suporte_Normal': int(tn + fp), 'suporte_Quebra': int(fn + tp),
        'precisao_Normal': p0, 'recall_Normal': r0, 'f1_Normal': f1_0,
        'precisao_Quebra': p1, 'recall_Quebra': r1, 'f1_Quebra': f1_1,
        'acuracia': (tn + tp) / len(y_true),
    }


def tabela_metricas(preds, y_true):
    return pd.DataFrame([metricas_por_classe(n, y_true, p) for n, p in preds.items()])


def formatar(df_, casas=2):
    def _f(v):
        if isinstance(v, (float, np.floating)):
            return 'n/d' if np.isnan(v) else f'{v:.{casas}f}'
        return v
    return df_.astype(object).apply(lambda col: col.map(_f))


def tabela_lift(m_s3, m_s4):
    # Delta S3 -> S4 para a classe positiva (Quebra). Linhas na mesma ordem de ORDEM_MODELOS.
    t = pd.DataFrame({
        'Modelo': m_s4['modelo'],
        'Recall S3': m_s3['recall_Quebra'], 'Recall S4': m_s4['recall_Quebra'],
        'F1 S3': m_s3['f1_Quebra'], 'F1 S4': m_s4['f1_Quebra'],
        'Precisão S3': m_s3['precisao_Quebra'], 'Precisão S4': m_s4['precisao_Quebra'],
        'FN S3': m_s3['FN'], 'FN S4': m_s4['FN'],
        'FP S3': m_s3['FP'], 'FP S4': m_s4['FP'],
        'Eventos reais': m_s4['suporte_Quebra'],
    })
    t['Delta recall'] = t['Recall S4'] - t['Recall S3']
    return t


print('Funções definidas.')

Funções definidas.


## 3. Lift no teste oficial (2023–2024)

S3 e S4 retreinados no mesmo split. Se o teste não tiver eventos de Quebra, recall, F1 e delta ficam `n/d`.

In [4]:
preds_s3, _ = prever_baselines(FEATURES_S3, df_train, df_test)
preds_s4, _ = prever_baselines(FEATURES_S4, df_train, df_test)
m_s3 = tabela_metricas(preds_s3, y_test)
m_s4 = tabela_metricas(preds_s4, y_test)

lift_teste = tabela_lift(m_s3, m_s4)
print('Lift S3 -> S4 no teste oficial (classe Quebra):')
display(formatar(lift_teste[['Modelo', 'Recall S3', 'Recall S4', 'F1 S3', 'F1 S4', 'Delta recall']]))

print('\nContagens absolutas (teste oficial):')
display(pd.concat([m_s3.assign(versao='S3'), m_s4.assign(versao='S4')])[
    ['versao', 'modelo', 'TN', 'FP', 'FN', 'TP', 'suporte_Quebra']].sort_values(['modelo', 'versao']))

if y_test.sum() == 0:
    print('\nO teste oficial não tem eventos de Quebra: o lift não é mensurável aqui (n/d), o que NÃO equivale a "sem ganho".')

# Conferência com o Tópico 5 (notebook 09)
caminho_t5 = '../models/s4_metricas_baselines_teste.csv'
if os.path.exists(caminho_t5):
    ref = pd.read_csv(caminho_t5)
    cols = ['TN', 'FP', 'FN', 'TP']
    assert (ref[cols].to_numpy() == m_s4[cols].to_numpy()).all(), 'S4 diverge do Tópico 5: reexecute o notebook 09'
    print('Conferido: as contagens S4 daqui são iguais às do Tópico 5 (notebook 09).')
else:
    print('Aviso: ../models/s4_metricas_baselines_teste.csv não encontrado; rode o notebook 09 para conferir.')

Lift S3 -> S4 no teste oficial (classe Quebra):


,Modelo,Recall S3,Recall S4,F1 S3,F1 S4,Delta recall
0,DummyClassifier,n/d,n/d,n/d,n/d,n/d
1,Persistência,n/d,n/d,n/d,n/d,n/d
2,GaussianNB,n/d,n/d,n/d,n/d,n/d



Contagens absolutas (teste oficial):


,versao,modelo,TN,FP,FN,TP,suporte_Quebra
0,S3,DummyClassifier,2,0,0,0,0
0,S4,DummyClassifier,2,0,0,0,0
2,S3,GaussianNB,2,0,0,0,0
2,S4,GaussianNB,2,0,0,0,0
1,S3,Persistência,2,0,0,0,0
1,S4,Persistência,2,0,0,0,0



O teste oficial não tem eventos de Quebra: o lift não é mensurável aqui (n/d), o que NÃO equivale a "sem ganho".
Conferido: as contagens S4 daqui são iguais às do Tópico 5 (notebook 09).


## 4. Lift no backtest temporal dentro do treino (2019–2022)

Janela expansiva: para cada safra-alvo, treina só com safras anteriores a ela; o teste oficial não é tocado.

**Limitações:** só 4 safras avaliadas e, em geral, poucos eventos; e a seleção das features S4 (notebook 08) usou a correlação com o alvo em **todo** o treino 2015–2022, então S4 sai favorecido. Use como indício, não como prova.

In [5]:
def backtest_expansivo(features):
    registros = []
    for ano_alvo in range(ANO_INICIO_BACKTEST, ANO_CORTE_TREINO + 1):
        tr = df_train[df_train['ano'] < ano_alvo]
        va = df_train[df_train['ano'] == ano_alvo]
        preds, _ = prever_baselines(features, tr, va)
        for nome, p in preds.items():
            registros.append({'modelo': nome, 'ano': ano_alvo,
                              'y_true': int(va[TARGET].iloc[0]), 'y_pred': int(p[0])})
    return pd.DataFrame(registros)


def metricas_backtest(bt):
    linhas = [metricas_por_classe(n, g['y_true'], g['y_pred']) for n, g in bt.groupby('modelo')]
    return pd.DataFrame(linhas).set_index('modelo').loc[ORDEM_MODELOS].reset_index()


def previsoes_por_safra(bt):
    visao = bt.pivot(index='ano', columns='modelo', values='y_pred')[ORDEM_MODELOS]
    visao.insert(0, 'real', bt.drop_duplicates('ano').set_index('ano')['y_true'])
    return visao


bt_s3, bt_s4 = backtest_expansivo(FEATURES_S3), backtest_expansivo(FEATURES_S4)

print('Previsões por safra no backtest — features S3 (0 = Normal, 1 = Quebra):')
display(previsoes_por_safra(bt_s3))
print('\nPrevisões por safra no backtest — features S4:')
display(previsoes_por_safra(bt_s4))

bt_m3, bt_m4 = metricas_backtest(bt_s3), metricas_backtest(bt_s4)
lift_bt = tabela_lift(bt_m3, bt_m4)

print('\nLift S3 -> S4 no backtest (classe Quebra)  [S4 otimista: ver limitações]')
display(formatar(lift_bt[['Modelo', 'Recall S3', 'Recall S4', 'F1 S3', 'F1 S4', 'Delta recall',
                          'Precisão S3', 'Precisão S4', 'FN S3', 'FN S4', 'FP S3', 'FP S4', 'Eventos reais']]))

Previsões por safra no backtest — features S3 (0 = Normal, 1 = Quebra):


modelo,real,DummyClassifier,Persistência,GaussianNB
ano,,,,
2019,0,0,0,1
2020,0,0,0,0
2021,1,0,0,0
2022,0,0,1,1



Previsões por safra no backtest — features S4:


modelo,real,DummyClassifier,Persistência,GaussianNB
ano,,,,
2019,0,0,0,1
2020,0,0,0,0
2021,1,0,0,1
2022,0,0,1,0



Lift S3 -> S4 no backtest (classe Quebra)  [S4 otimista: ver limitações]


,Modelo,Recall S3,Recall S4,F1 S3,F1 S4,Delta recall,Precisão S3,Precisão S4,FN S3,FN S4,FP S3,FP S4,Eventos reais
0,DummyClassifier,0.00,0.00,0.00,0.00,0.00,n/d,n/d,1,1,0,0,1
1,Persistência,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1,1,1,1,1
2,GaussianNB,0.00,1.00,0.00,0.67,1.00,0.00,0.50,1,0,2,1,1


## 5. Sensibilidade do ganho do Naive Bayes

O ganho resiste a duas mudanças?
1. **S4 sem `amplitude_termica`** (feature com vazamento potencial; ver seção 6).
2. **Seleção refeita dentro de cada dobra**: em cada safra-alvo, as 4 variáveis de maior |correlação| com o alvo, usando só as safras anteriores (mais simples que o notebook 08: sem filtro de redundância nem critério agronômico).

In [6]:
def backtest_nb_selecao_na_dobra(k=4):
    linhas = []
    for ano_alvo in range(ANO_INICIO_BACKTEST, ANO_CORTE_TREINO + 1):
        tr = df_train[df_train['ano'] < ano_alvo]
        va = df_train[df_train['ano'] == ano_alvo]
        corr = tr[CANDIDATAS + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
        feats = corr.index[:k].tolist()
        pipe = criar_pipeline(GaussianNB(), feats).fit(tr[feats], tr[TARGET])
        linhas.append({'ano': ano_alvo, 'y_true': int(va[TARGET].iloc[0]),
                       'y_pred': int(pipe.predict(va[feats])[0]), 'features': feats})
    return pd.DataFrame(linhas)


def linha_nb(rotulo, bt):
    g_ = bt.sort_values('ano')
    m = metricas_por_classe('GaussianNB', g_['y_true'], g_['y_pred'])
    return {'configuração': rotulo, 'TP': m['TP'], 'FN': m['FN'], 'FP': m['FP'], 'TN': m['TN'],
            'precisão Quebra': m['precisao_Quebra'], 'recall Quebra': m['recall_Quebra'], 'F1 Quebra': m['f1_Quebra']}


bt_sem_amp = backtest_expansivo([f for f in FEATURES_S4 if f != 'amplitude_termica'])
bt_selecao_dobra = backtest_nb_selecao_na_dobra(k=4)

sens = pd.DataFrame([
    linha_nb('S3 (9 features)', bt_s3[bt_s3['modelo'] == 'GaussianNB']),
    linha_nb('S4 (4 features, seleção no treino todo)', bt_s4[bt_s4['modelo'] == 'GaussianNB']),
    linha_nb('S4 sem amplitude_termica (3 features)', bt_sem_amp[bt_sem_amp['modelo'] == 'GaussianNB']),
    linha_nb('Top-4 |r| escolhidas dentro de cada dobra', bt_selecao_dobra),
])
print('GaussianNB no backtest 2019–2022 (classe Quebra):')
display(formatar(sens))

print('\nFeatures escolhidas em cada dobra (top-4 |r|):')
display(bt_selecao_dobra[['ano', 'y_true', 'y_pred', 'features']])

GaussianNB no backtest 2019–2022 (classe Quebra):


,configuração,TP,FN,FP,TN,precisão Quebra,recall Quebra,F1 Quebra
0,S3 (9 features),0,1,2,1,0.00,0.00,0.00
1,"S4 (4 features, seleção no treino todo)",1,0,1,2,0.50,1.00,0.67
2,S4 sem amplitude_termica (3 features),1,0,1,2,0.50,1.00,0.67
3,Top-4 |r| escolhidas dentro de cada dobra,0,1,1,2,0.00,0.00,0.00



Features escolhidas em cada dobra (top-4 |r|):


,ano,y_true,y_pred,features
0,2019,0,1,"[deficit_hidrico_solo, T2M_MIN, amplitude_term..."
1,2020,0,0,"[deficit_hidrico_solo, amplitude_termica, T2M,..."
2,2021,1,0,"[deficit_hidrico_solo, amplitude_termica, T2M_..."
3,2022,0,0,"[deficit_hidrico_solo, dias_pico_calor, verani..."


## 6. Fatos agronômicos que sustentam a leitura e checagem do vazamento

Aqui ficam reproduzidos os números citados na interpretação: estresse de janeiro–fevereiro por safra, correlações no treino e o mês em que ocorrem os extremos térmicos anuais (base de `amplitude_termica`).

In [7]:
fatos = df[['ano', TARGET, 'PRECTOTCORR', 'deficit_hidrico_solo', 'dias_pico_calor',
            'precipitacao_enchimento', 'amplitude_termica']].rename(columns={'PRECTOTCORR': 'chuva_anual_mm'})
print('Safras: alvo, chuva anual e estresse de janeiro–fevereiro')
display(fatos)

ranking_chuva = df.sort_values('PRECTOTCORR', ascending=False)['ano'].tolist()
print(f"\nRanking de chuva anual (maior -> menor): {ranking_chuva}")
print(f"Posição de 2021: {ranking_chuva.index(2021) + 1}ª maior chuva anual")

corr_alvo = df_train[CANDIDATAS + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
print('\nCorrelação com quebra_safra no treino (N = 8):')
display(corr_alvo.round(2).to_frame('r'))

print('Correlação entre as features de janeiro–fevereiro (treino):')
display(df_train[['deficit_hidrico_solo', 'dias_pico_calor', 'precipitacao_enchimento']].corr().round(2))
print('Correlação entre as features anuais descartadas (treino):')
display(df_train[['PRECTOTCORR', 'GWETROOT', 'GWET_x_PREC']].corr().round(3))

Safras: alvo, chuva anual e estresse de janeiro–fevereiro


,ano,quebra_safra,chuva_anual_mm,deficit_hidrico_solo,dias_pico_calor,precipitacao_enchimento,amplitude_termica
0,2015,1,645.49,47,0,159.28,24.48
1,2016,1,1147.75,28,4,257.40,23.79
2,2017,0,1561.12,0,0,425.38,27.62
3,2018,0,1903.15,0,0,724.46,26.29
4,2019,0,1210.35,15,0,382.11,25.44
5,2020,0,977.14,15,0,299.01,27.58
6,2021,1,1634.39,42,3,418.57,28.69
7,2022,0,1518.05,0,0,463.11,30.48
8,2023,0,1250.03,0,0,476.27,27.11
9,2024,0,1469.91,4,0,556.25,27.50



Ranking de chuva anual (maior -> menor): [2018, 2021, 2017, 2022, 2024, 2023, 2019, 2016, 2020, 2015]
Posição de 2021: 2ª maior chuva anual

Correlação com quebra_safra no treino (N = 8):


,r
precipitacao_enchimento,-0.55
GWETROOT,-0.54
RH2M,-0.51
amplitude_termica,-0.42
GWET_x_PREC,-0.40
precipitacao_por_temperatura,-0.39
PRECTOTCORR,-0.37
T2M_MAX,0.25
T2M_MIN,0.46
veranico_max_dias,0.53


Correlação entre as features de janeiro–fevereiro (treino):


,deficit_hidrico_solo,dias_pico_calor,precipitacao_enchimento
deficit_hidrico_solo,1.00,0.50,-0.67
dias_pico_calor,0.50,1.00,-0.23
precipitacao_enchimento,-0.67,-0.23,1.00


Correlação entre as features anuais descartadas (treino):


,PRECTOTCORR,GWETROOT,GWET_x_PREC
PRECTOTCORR,1.000,0.933,0.996
GWETROOT,0.933,1.000,0.959
GWET_x_PREC,0.996,0.959,1.000


In [8]:
# Vazamento potencial em amplitude_termica: em que mês ocorrem o mínimo e o máximo anuais?
assert (df['T2M_MAX'] - df['T2M_MIN'] - df['amplitude_termica']).abs().max() < 1e-6, 'amplitude != T2M_MAX - T2M_MIN'

with open('../data/raw/nasa_power_Sorriso_MT_2015_2025.json') as f:
    raw = json.load(f)
par = raw['properties']['parameter']
diario = pd.DataFrame({'t2m_min': par['T2M_MIN'], 't2m_max': par['T2M_MAX']})
diario.index = pd.to_datetime(diario.index, format='%Y%m%d')
diario = diario.replace(-999.0, np.nan)
diario = diario[(diario.index.year >= 2015) & (diario.index.year <= 2024)]
por_ano = diario.index.year

extremos = pd.DataFrame({
    't2m_min_anual': diario.groupby(por_ano)['t2m_min'].min(),
    'mes_do_minimo': diario.groupby(por_ano)['t2m_min'].idxmin().dt.month,
    't2m_max_anual': diario.groupby(por_ano)['t2m_max'].max(),
    'mes_do_maximo': diario.groupby(por_ano)['t2m_max'].idxmax().dt.month,
})
extremos['fora_de_jan_fev'] = ~extremos['mes_do_minimo'].isin([1, 2]) & ~extremos['mes_do_maximo'].isin([1, 2])
display(extremos)
print(f"\nSafras em que mínimo E máximo anuais caem fora de janeiro–fevereiro: {int(extremos['fora_de_jan_fev'].sum())} de {len(extremos)}")
print('Conclusão: amplitude_termica usa informação posterior à janela de decisão (jan–fev) -> vazamento potencial.')

,t2m_min_anual,mes_do_minimo,t2m_max_anual,mes_do_maximo,fora_de_jan_fev
2015,17.31,7,41.79,10,True
2016,15.78,4,39.57,9,True
2017,12.24,7,39.86,10,True
2018,13.42,7,39.71,9,True
2019,15.47,7,40.91,9,True
2020,14.45,5,42.03,10,True
2021,12.56,7,41.25,9,True
2022,8.74,5,39.22,9,True
2023,14.45,6,41.56,11,True
2024,14.04,7,41.54,10,True



Safras em que mínimo E máximo anuais caem fora de janeiro–fevereiro: 10 de 10
Conclusão: amplitude_termica usa informação posterior à janela de decisão (jan–fev) -> vazamento potencial.


## 7. Tabelas em Markdown (para `docs/sprints/Sprint4_TrilhaC.md`) e artefatos

In [9]:
def pt(v, casas=2):
    return 'n/d' if pd.isna(v) else f'{v:.{casas}f}'.replace('.', ',')


def md_lift_teste(t):
    linhas = ['| Modelo | Recall S3 | Recall S4 | F1 S3 | F1 S4 | Delta recall |', '|---|---|---|---|---|---|']
    for _, r in t.iterrows():
        linhas.append(f"| {r['Modelo']} | {pt(r['Recall S3'])} | {pt(r['Recall S4'])} | {pt(r['F1 S3'])} | {pt(r['F1 S4'])} | {pt(r['Delta recall'])} |")
    return '\n'.join(linhas)


def md_lift_backtest(t):
    linhas = ['| Modelo | Recall S3 | Recall S4 | F1 S3 | F1 S4 | Delta recall | FN S3 → S4 | FP S3 → S4 |',
              '|---|---|---|---|---|---|---|---|']
    for _, r in t.iterrows():
        ev = int(r['Eventos reais'])
        tp3, tp4 = ev - int(r['FN S3']), ev - int(r['FN S4'])
        linhas.append(
            f"| {r['Modelo']} | {pt(r['Recall S3'])} ({tp3} de {ev}) | {pt(r['Recall S4'])} ({tp4} de {ev}) | "
            f"{pt(r['F1 S3'])} | {pt(r['F1 S4'])} | {pt(r['Delta recall'])} | "
            f"{int(r['FN S3'])} → {int(r['FN S4'])} | {int(r['FP S3'])} → {int(r['FP S4'])} |")
    return '\n'.join(linhas)


print('--- Teste oficial (2023–2024) ---')
print(md_lift_teste(lift_teste))
print('\n--- Backtest 2019–2022 ---')
print(md_lift_backtest(lift_bt))

os.makedirs('../models', exist_ok=True)
lift_teste.to_csv('../models/s4_lift_teste.csv', index=False)
lift_bt.to_csv('../models/s4_lift_backtest.csv', index=False)
sens.to_csv('../models/s4_sensibilidade_backtest_nb.csv', index=False)
print('\nSalvos em ../models/: s4_lift_teste.csv, s4_lift_backtest.csv, s4_sensibilidade_backtest_nb.csv')

--- Teste oficial (2023–2024) ---
| Modelo | Recall S3 | Recall S4 | F1 S3 | F1 S4 | Delta recall |
|---|---|---|---|---|---|
| DummyClassifier | n/d | n/d | n/d | n/d | n/d |
| Persistência | n/d | n/d | n/d | n/d | n/d |
| GaussianNB | n/d | n/d | n/d | n/d | n/d |

--- Backtest 2019–2022 ---
| Modelo | Recall S3 | Recall S4 | F1 S3 | F1 S4 | Delta recall | FN S3 → S4 | FP S3 → S4 |
|---|---|---|---|---|---|---|---|
| DummyClassifier | 0,00 (0 de 1) | 0,00 (0 de 1) | 0,00 | 0,00 | 0,00 | 1 → 1 | 0 → 0 |
| Persistência | 0,00 (0 de 1) | 0,00 (0 de 1) | 0,00 | 0,00 | 0,00 | 1 → 1 | 1 → 1 |
| GaussianNB | 0,00 (0 de 1) | 1,00 (1 de 1) | 0,00 | 0,67 | 1,00 | 1 → 0 | 2 → 1 |

Salvos em ../models/: s4_lift_teste.csv, s4_lift_backtest.csv, s4_sensibilidade_backtest_nb.csv


## 8. Conferência dos números citados na interpretação

Se os dados mudarem (por exemplo, o PO pedir outro corte ou outras features), esta célula quebra e avisa que o texto da seção 9 precisa ser revisto.

In [10]:
def linha(t, modelo):
    return t[t['Modelo'] == modelo].iloc[0]

# Teste oficial sem eventos de Quebra
assert (lift_teste['Eventos reais'] == 0).all(), 'O teste agora tem eventos de Quebra: revisar a seção 9'

# Backtest: Naive Bayes S3 -> S4
b = linha(lift_bt, 'GaussianNB')
assert (b['Eventos reais'], b['FN S3'], b['FP S3'], b['FN S4'], b['FP S4']) == (1, 1, 2, 0, 1)
assert round(b['F1 S4'], 2) == 0.67 and round(b['Precisão S4'], 2) == 0.50 and b['Recall S3'] == 0 and b['Recall S4'] == 1
# Dummy e persistência não mudam
for nome in ['DummyClassifier', 'Persistência']:
    r = linha(lift_bt, nome)
    assert r['Delta recall'] == 0 and r['FN S3'] == r['FN S4'] and r['FP S3'] == r['FP S4']
pers = previsoes_por_safra(bt_s4)
assert pers.loc[2021, 'Persistência'] == 0 and pers.loc[2022, 'Persistência'] == 1
# Sensibilidade
s = sens.set_index('configuração')
assert (s.loc['S4 sem amplitude_termica (3 features)', ['TP', 'FP']] == [1, 1]).all()
assert (s.loc['Top-4 |r| escolhidas dentro de cada dobra', ['TP', 'FN', 'FP']] == [0, 1, 1]).all()
# Fatos agronômicos
f = df.set_index('ano')
assert f.loc[2021, 'deficit_hidrico_solo'] == 42 and f.loc[2021, 'dias_pico_calor'] == 3
assert f.loc[2015, 'deficit_hidrico_solo'] == 47 and round(f.loc[2015, 'precipitacao_enchimento'], 2) == 159.28
assert all(f.loc[a, 'deficit_hidrico_solo'] == 0 for a in (2017, 2018, 2022))
assert f['precipitacao_enchimento'].idxmin() == 2015
assert ranking_chuva.index(2021) == 1 and round(f.loc[2021, 'PRECTOTCORR'], 2) == 1634.39
assert round(corr_alvo['deficit_hidrico_solo'], 2) == 0.90 and round(corr_alvo['dias_pico_calor'], 2) == 0.74
assert round(corr_alvo['precipitacao_enchimento'], 2) == -0.55 and round(corr_alvo['veranico_max_dias'], 2) == 0.53
c3 = df_train[['deficit_hidrico_solo', 'dias_pico_calor', 'precipitacao_enchimento']].corr()
assert round(c3.loc['deficit_hidrico_solo', 'dias_pico_calor'], 2) == 0.50
assert round(c3.loc['deficit_hidrico_solo', 'precipitacao_enchimento'], 2) == -0.67
assert df_train[['PRECTOTCORR', 'GWETROOT', 'GWET_x_PREC']].corr().to_numpy()[np.triu_indices(3, 1)].min() > 0.93
assert extremos['fora_de_jan_fev'].all()
print('OK: todos os números citados na interpretação (seção 9) foram conferidos.')

OK: todos os números citados na interpretação (seção 9) foram conferidos.


## 9. Interpretação do lift (ou da ausência de lift)

**1. Teste oficial: o lift não é mensurável.** As safras de 2023 (3.750 kg/ha) e 2024 (3.480 kg/ha) ficaram acima do limiar de 3.400 kg/ha. Os três baselines acertam as duas em S3 e em S4, então a acurácia de 100% não diz nada sobre risco. Isso é ausência de medida, não ausência de ganho: o teste não consegue comprovar nem refutar o lift. Seguindo a regra do projeto, o teste não foi alterado.

**2. Backtest: ganho só no Naive Bayes, e por motivo claro.** O Dummy ignora as features (prevê sempre Normal, pois o treino tem 62,5% de safras normais). A Persistência usa apenas o rótulo da safra anterior, então seu resultado não muda de S3 para S4 (delta = 0 por construção): ela erra 2021 (2020 foi normal) e dá falso positivo em 2022 (herdou a quebra de 2021). O Naive Bayes com features S3 não sinalizou a quebra de 2021 (FN) e gerou 2 falsos positivos (2019 e 2022); com S4 sinalizou 2021 e ficou com 1 falso positivo (2019).

**3. Evolução agronômica.** As features S3 eram totais do ano-calendário e escondiam o estresse da floração e do enchimento de grãos. Em 2021 a chuva anual foi de 1.634,39 mm (2ª maior da série 2015–2024), mas no bimestre janeiro–fevereiro houve 42 dias com `GWETROOT < 0,70` (contra 0 dias em 2017, 2018 e 2022) e 3 dias com `T2M_MAX ≥ 35 °C`. Em 2015 foram 47 dias de solo seco e só 159,28 mm em janeiro–fevereiro, o menor volume da série. No treino (N = 8), a correlação com `quebra_safra` é de +0,90 para `deficit_hidrico_solo`, +0,74 para `dias_pico_calor` e −0,55 para `precipitacao_enchimento`. A leitura é que o déficit hídrico na janela crítica (premissa agronômica adotada no notebook 08) separa melhor safras de risco do que o volume anual de chuva.

**4. Por que isso ainda não é prova de ganho.**
- O backtest tem um único evento: uma previsão a mais ou a menos muda o recall de 0 para 1.
- As features S4 foram desenhadas depois de observar o erro de 2021 e selecionadas pela correlação no treino inteiro (que inclui as safras avaliadas no backtest), então o resultado é otimista por construção.
- Sensibilidade (seção 5 deste notebook): sem `amplitude_termica` o resultado do Naive Bayes é idêntico (TP = 1, FP = 1), então o ganho não vem da feature com vazamento. Porém, refazendo a seleção dentro de cada dobra (4 variáveis de maior |r| com as safras anteriores, sem filtro agronômico), o Naive Bayes volta a errar 2021 (TP = 0, FN = 1, FP = 1). O ganho depende da escolha das variáveis de janeiro–fevereiro, feita com conhecimento de 2021.
- **Vazamento em `amplitude_termica`:** ela é `T2M_MAX − T2M_MIN` do ano-calendário inteiro. Em 2015–2024 o mínimo anual ocorreu entre abril e julho e o máximo entre setembro e novembro, depois da janela de decisão (janeiro–fevereiro). Como o ganho não depende dela, removê-la ou recalculá-la só com janeiro–fevereiro tem custo baixo, mas qualquer mudança no transformer congelado exige retreinar os três baselines.
- Independência condicional: as variáveis de janeiro–fevereiro ainda são correlacionadas no treino (r = +0,50 entre déficit hídrico e dias de calor; r = −0,67 entre déficit hídrico e chuva de janeiro–fevereiro). A premissa do Naive Bayes segue violada, ainda que menos do que nas variáveis anuais (r > 0,93 entre `PRECTOTCORR`, `GWETROOT` e `GWET_x_PREC`).

**5. Conclusão e o que foi descartado.** Há um **ganho sugerido, mas não comprovado**: no backtest, o Naive Bayes passa de 0 de 1 para 1 de 1 evento detectado, com agronomia coerente por trás. O teste oficial não permite confirmar. O conjunto S4 segue congelado para a Sprint 5 por ser parcimonioso e justificável, mas o lift deve ser tratado como hipótese a validar com a validação temporal no fim do treino e o limiar da Sprint 5. Foram descartadas na seleção (notebook 08): `PRECTOTCORR`, `GWETROOT` anual e `GWET_x_PREC` (colineares entre si e mais fracas); `T2M_MAX` e `T2M_MIN` brutas (redundantes com a amplitude); e `precipitacao_por_temperatura` (redundante com `precipitacao_enchimento`). `veranico_max_dias` (r = +0,53) foi calculada, mas não entrou na lista final.

## 10. Sprint Review — incremento demonstrado e feedback

**Incremento demonstrado ao PO (docente):**

- Atributos novos de janeiro–fevereiro (`deficit_hidrico_solo`, `dias_pico_calor`, `precipitacao_enchimento`) mais `amplitude_termica` herdada da Sprint 2, selecionados só no treino (N = 8) — notebook 08.
- Pipeline congelado (`ColumnTransformer` com `StandardScaler`, `fit` só no treino) e os três baselines retreinados no mesmo split da Sprint 2/3 — notebook 09, com artefatos em `models/`.
- Tabela de lift S3 → S4 e sua leitura (notebook 10): o teste oficial (2023–2024) não tem eventos de quebra, então o lift não é mensurável ali. No backtest 2019–2022, o Naive Bayes passa de recall 0 de 1 para 1 de 1 e F1 de 0,00 para 0,67; Dummy e Persistência não mudam.
- Pontos levantados para decisão: o ganho é exploratório (1 evento; sensível à seleção das variáveis) e `amplitude_termica` tem vazamento (usa o ano-calendário inteiro).

**Feedback recebido:**

_A preencher após a Sprint Review (data da reunião, o que o PO disse, decisões tomadas)._

Sugestão de pauta para decisão com o PO:
1. Manter `amplitude_termica` no transformer congelado ou removê-la/recalculá-la só com janeiro–fevereiro (isso exige retreinar os três baselines).
2. Aceitar o backtest dentro do treino como evidência complementar de lift, já que o teste oficial não tem quebras.
3. Confirmar que a Sprint 5 mantém o teste 2023–2024 sem alteração e usa validação temporal no fim do treino para o limiar.